# Football Ball Detector — YOLO11m fine-tune

Trains a single-class (`football`) detector on 288 auto-labeled drill frames
(Grounding DINO labels, size-filtered + NMS-deduped).

Balls are ~20 px, so `imgsz=1280` is load-bearing — do not lower it.

**Runtime → Change runtime type → T4 GPU** (or better) before running.

In [ ]:
# 1. Confirm we actually have a GPU (this whole notebook exists because MPS ran out of memory)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# 2. Install
!pip -q install ultralytics
import torch, ultralytics
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
print("ultralytics", ultralytics.__version__)

In [ ]:
# 3. Get the dataset zip into Colab. Two ways — pick one.

# OPTION A — browser upload. Simple, but 133 MB over files.upload() is slow and
# sometimes dies partway. If it stalls, use Option B.
from google.colab import files
up = files.upload()          # pick football_yolo_dataset.zip
ZIP = "football_yolo_dataset.zip"

# OPTION B — Google Drive (more reliable). Put football_yolo_dataset.zip in the
# top level of your Drive first, then comment out Option A above and run this:
# from google.colab import drive
# drive.mount("/content/drive")
# ZIP = "/content/drive/MyDrive/football_yolo_dataset.zip"

print("zip =", ZIP)

In [ ]:
# 4. Unzip, then REWRITE data.yaml for Colab's filesystem.
#    Ultralytics resolves a relative `path:` against its own datasets dir, not
#    against the yaml's location — so `path:` must be an absolute Colab path.
!unzip -q -o "$ZIP" -d /content/

from pathlib import Path
root = Path("/content/dataset/yolo")
(root / "data.yaml").write_text(
    f"path: {root}\ntrain: images/train\nval: images/val\nnames:\n  0: football\n"
)

for split in ("train", "val"):
    lbls = sorted((root / "labels" / split).glob("*.txt"))
    n_img = len(list((root / "images" / split).glob("*.jpg")))
    n_box = sum(len([l for l in f.read_text().split("\n") if len(l.split()) == 5]) for f in lbls)
    n_empty = sum(1 for f in lbls if not f.read_text().strip())
    print(f"{split}: {n_img} images, {len(lbls)} labels, {n_box} boxes, {n_empty} empty")

print()
print((root / "data.yaml").read_text())

In [ ]:
# 5. Eyeball one labeled frame before burning GPU time on it
import cv2, random
from pathlib import Path
import matplotlib.pyplot as plt

random.seed(0)
p = random.choice(sorted((root/"images"/"train").glob("*.jpg")))
img = cv2.imread(str(p)); h, w = img.shape[:2]
for line in (root/"labels"/"train"/f"{p.stem}.txt").read_text().split("\n"):
    f = line.split()
    if len(f) != 5: continue
    cx, cy, bw, bh = (float(v) for v in f[1:])
    cx, cy, bw, bh = cx*w, cy*h, bw*w, bh*h
    cv2.rectangle(img, (int(cx-bw/2), int(cy-bh/2)), (int(cx+bw/2), int(cy+bh/2)), (0,255,0), 3)
plt.figure(figsize=(15,8)); plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.title(p.name); plt.axis("off"); plt.show()

In [ ]:
# 6. Train — same hyperparameters as pipeline/04_train.py, device swapped mps -> cuda.
#    T4 (16 GB) handles batch=8 at 1280. On an A100 you can raise to 16.
from ultralytics import YOLO

model = YOLO("yolo11m.pt")
results = model.train(
    data="/content/dataset/yolo/data.yaml",
    epochs=60,
    imgsz=1280,
    batch=8,
    device=0,
    patience=15,
    cos_lr=True,
    mosaic=1.0,
    mixup=0.1,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=5, translate=0.1, scale=0.5, fliplr=0.5,
    project="/content/runs",
    name="football_yolo11m",
    exist_ok=True,
)

In [ ]:
# 7. The numbers worth quoting
m = model.val(data="/content/dataset/yolo/data.yaml", imgsz=1280, device=0)
print(f"mAP50    : {m.box.map50:.4f}")
print(f"mAP50-95 : {m.box.map:.4f}")
print(f"precision: {m.box.mp:.4f}")
print(f"recall   : {m.box.mr:.4f}")

In [ ]:
# 8. Training curves + val predictions
from IPython.display import Image, display
for f in ["results.png", "val_batch0_pred.jpg", "confusion_matrix_normalized.png"]:
    p = f"/content/runs/football_yolo11m/{f}"
    import os
    if os.path.exists(p):
        print(f); display(Image(p, width=1000))

In [ ]:
# 9. Pull the weights back down -> drop into pipeline/ as football_best.pt
from google.colab import files
files.download("/content/runs/football_yolo11m/weights/best.pt")